In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
# Helper functions
from anthropic.types import Message

# Note: the magic string that triggered RedactedThinkingBlock in the original
# course notebook no longer produces that block type on current models.
# RedactedThinkingBlock was specific to the budget_tokens thinking API
# (thinking.type: "enabled"), which is deprecated on Sonnet/Opus 4.6 and
# removed on Opus 4.7+. The modern replacement is adaptive thinking.


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_effort="high",
):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        # Adaptive thinking: Claude decides if and when to think based on
        # problem complexity. effort controls depth: low, medium, high, max.
        # budget_tokens (thinking.type: "enabled") is deprecated on 4.6 and
        # removed on Opus 4.7+ — use adaptive thinking instead.
        params["thinking"] = {
            "type": "adaptive",
            "effort": thinking_effort,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def thinking_from_message(message):
    """Extract thinking content from a message response.
    With adaptive thinking, Claude may or may not produce a thinking block
    depending on problem complexity. Returns None if no thinking block present.
    """
    for block in message.content:
        if block.type == "thinking":
            return block.thinking
    return None

In [ ]:
# Demonstrate adaptive thinking on a complex multi-step reasoning problem.
# A simple question may not trigger a thinking block at all — adaptive thinking
# means Claude decides when reasoning is needed based on problem complexity.
# Use a genuinely hard problem to reliably produce a ThinkingBlock.

messages = []

add_user_message(
    messages,
    """
    A client has three business units: Unit A generates $12M revenue with 40% margin,
    Unit B generates $8M revenue with 60% margin, Unit C generates $20M revenue with
    15% margin. They have $5M to invest in exactly one unit to improve its margin by
    10 percentage points. Which unit maximizes total profit improvement, and what is
    the incremental profit gain from that investment?
    """,
)

response = chat(messages, thinking=True, thinking_effort="high")

# Inspect the full response structure
print("Content block types:", [block.type for block in response.content])
print()

# Extract and display thinking if present
thinking_content = thinking_from_message(response)
if thinking_content:
    print("=== THINKING ===")
    print(thinking_content)
    print()

print("=== RESPONSE ===")
print(text_from_message(response))